# Resource estimation for the 2D Fermi-Hubbard model

This notebook estimates the cost of finding the ground-state energy of the two-dimensional
Fermi-Hubbard model by quantum phase estimation, using the plaquette Trotterization of the
time evolution.

It has two halves:

1. a **logical** sweep over lattice sizes, which counts qubits, rotations, T gates and
   Toffolis in the emitted circuit; and
2. a **physical** estimate for one lattice, which maps those logical counts onto a
   fault-tolerant architecture and reports physical qubits and runtime.

The logical counts are pinned by a regression test in
`python/tests/test_hubbard_plaquette_trotter.py`, which builds the same circuits directly
through the library, so the numbers this notebook produces are checked rather than merely
displayed.

> The import cell silences QDK/Chemistry library logs so the tables stay readable. Change
> `Logger.LogLevel.off` to `Logger.LogLevel.info` and rerun that cell to see detailed
> calculation logs.

## Background: the 2D Fermi-Hubbard model

The Fermi-Hubbard Hamiltonian on a square lattice is

$$
H = -t \sum_{\langle i,j \rangle, \sigma} \left( c^\dagger_{i\sigma} c_{j\sigma} + \text{h.c.} \right)
  + U \sum_i n_{i\uparrow} n_{i\downarrow},
$$

a nearest-neighbour hopping term of amplitude $t$ and an on-site repulsion $U$ paid whenever
a site holds both spin species. Each of the $L \times L$ sites carries two spin orbitals, so
an $L \times L$ lattice needs $2L^2$ qubits under the Jordan-Wigner mapping.

We take $U = 8t$, the strong-coupling regime where the model is least classically tractable
and the ground state is most interesting, and a filling of $0.875$ electrons per site — near
but not at half filling, which is the regime relevant to the cuprate phase diagram. The
lattice is periodic in both directions.

In [ ]:
import math

import pandas as pd
from qdk import qsharp
from qdk_chemistry.algorithms import create
from qdk_chemistry.algorithms.state_preparation import identity_state_prep
from qdk_chemistry.data import AlgorithmRef, LatticeGraph, QubitOperator
from qdk_chemistry.data.qubit_operator.containers.lattice import LatticeContainer
from qdk_chemistry.utils import Logger
from qdk_chemistry.utils.qsharp import create_qsharp_context, use_qsharp_context

# Silence the library logs so the tables below stay readable.
Logger.set_global_level(Logger.LogLevel.off)

#: Hopping amplitude, which sets the unit of energy.
HOPPING_T = 1.0

#: U = 8t for the strong-coupling regime.
U_OVER_T = 8.0

#: Electrons per SITE, not per orbital.
FILLING = 0.875

## The error budget

Three independent error sources have to share one energy budget: the phase-estimation
readout, the Trotter splitting, and the synthesis of the rotation angles into a discrete
gate set. We allow

$$
\epsilon_{\text{total}} = 0.0051 \ \text{per site},
$$

so the budget grows with the lattice rather than being a fixed absolute accuracy. A fixed
absolute target would make large lattices arbitrarily expensive without making the answer
more physically meaningful, since the quantity of interest is the energy density.

**How the budget is split.** Give a fraction $f$ of the energy budget to phase estimation
and the remaining $1 - f$ to Trotter error. The total number of Trotter steps summed over
the phase-estimation queries, $\sum_k r_k$, then scales as

$$
\sum_k r_k \ \sim \ \frac{1}{f \sqrt{1 - f}},
$$

because the query depth scales as $1/f$ while the step count per query scales as
$1/\sqrt{1-f}$ for a second-order product formula. Minimizing that expression over $f$ gives

$$
f = \tfrac{2}{3},
$$

which is the same optimum Campbell uses (arXiv:2012.09238v4, App. F). So two thirds of the
budget goes to phase estimation and one third to Trotter error.

In [ ]:
#: Total error budget per site, covering phase estimation, Trotter and gate synthesis.
TARGET_PRECISION_PER_SITE = 0.0051

#: Number of phase qubits in QPE.
QPE_PRECISION_BITS = 10

#: Share of the energy budget allocated to phase estimation. Minimizing the total Trotter
#: step count sum_k r_k, which scales as 1 / (f * sqrt(1 - f)) in the phase-estimation
#: share f, gives f = 2/3. Same optimum used in Campbell (arXiv:2012.09238v4, App. F).
QPE_BUDGET_FRACTION = 2.0 / 3.0

#: Plaquette Trotter order.
TROTTER_ORDER = 2

## Building the lattice and the qubit operator

The lattice is carried by a `LatticeContainer` rather than an expanded list of Pauli terms.
That matters for the large sizes below: the operator's representation stays $O(1)$ in the
lattice size, so nothing in this notebook ever materializes the $O(L^2)$ terms of the
Hamiltonian in Python.

In [ ]:
#: Lattice used for the narrative below and for the physical estimate at the end.
EXAMPLE_SIZE = 4


def build_lattice_operator(size):
    """Return the periodic square lattice of ``size`` x ``size`` sites as a qubit operator."""
    lattice = LatticeGraph.square(size, size, periodic_x=True, periodic_y=True)
    return QubitOperator(container=LatticeContainer(lattice))


operator = build_lattice_operator(EXAMPLE_SIZE)
print(f"{EXAMPLE_SIZE}x{EXAMPLE_SIZE} lattice: {EXAMPLE_SIZE**2} sites, {operator.num_qubits} qubits")
print(f"electrons at filling {FILLING}: {round(FILLING * EXAMPLE_SIZE**2)}")

## How long each phase-estimation query evolves for

Phase estimation reads the phase $\tau E$ accumulated by evolving for a time $\tau$, so the
energy resolution is set by how long the evolution runs. We prepare the phase register in a
**sine window** rather than a uniform superposition: with $N = 2^b - 1$ queries the sine
window's phase estimate has spread

$$
\tan\!\left(\frac{\pi}{N + 2}\right),
$$

which is Heisenberg-limited and a constant factor better than the uniform window. Setting
that spread equal to the required $\epsilon_{\text{QPE}} \cdot \tau$ fixes the base evolution
time.

Bit $k$ of the phase register then evolves for `base_time` $\cdot\, 2^k$ — the `"rescale"`
power strategy — rather than repeating the evolution block $2^k$ times. Rescaling the time
inside a single block is far cheaper than repeating the block, and is exact here because the
Trotter step count is re-derived for each rescaled time.

In [ ]:
def evolution_schedule(size):
    """Return the energy budgets and the base evolution time for one lattice size."""
    num_sites = size * size
    energy_budget = TARGET_PRECISION_PER_SITE * num_sites
    qpe_budget = QPE_BUDGET_FRACTION * energy_budget
    trotter_budget = energy_budget - qpe_budget
    # A sine-windowed QPE phase state of N = 2^bits - 1 queries has spread
    # tan(pi / (N + 2)), which equals the required eps_QPE * tau.
    base_time = math.tan(math.pi / (2**QPE_PRECISION_BITS - 1 + 2)) / qpe_budget
    return energy_budget, qpe_budget, trotter_budget, base_time


energy_budget, qpe_budget, trotter_budget, base_time = evolution_schedule(EXAMPLE_SIZE)
print(f"energy budget:  {energy_budget:.4f}")
print(f"  phase est.:   {qpe_budget:.4f}  ({QPE_BUDGET_FRACTION:.3f} of the budget)")
print(f"  Trotter:      {trotter_budget:.4f}")
print(f"base time:      {base_time:.6f}")
print(f"longest query:  {base_time * 2**QPE_PRECISION_BITS:.4f}")

## Building the time-evolution circuit

The evolution uses the **plaquette Trotterization**: the lattice is tiled twice by
four-cycles (Campbell's "pink" and "gold" tilings), each plaquette is rotated into the
momentum basis by a fermionic FFT, and the hopping term becomes diagonal there. Combined
with the on-site interaction layer, one second-order Trotter step is a symmetric product of
four layers.

Every layer ends up as a batch of **same-angle** $R_z$ rotations. Once a batch reaches eight
terms, it is phased through a Hamming-weight register instead of term by term: an adder tree
compresses $n$ same-angle rotations into a $\log_2 n$-bit weight, and only one rotation per
place value is applied. This is the construction of :cite:`Kan2025`, whose Methods
diagonalize every plaquette and every on-site pair into a layer of same-angle $R_z$ gates and
synthesize that layer collectively.

The break-even at eight terms is deliberate and measured: below it the adder tree costs more
Toffolis than the rotations it saves. **This is why the $L = 2$ row below has zero Toffolis
and every larger row has many** — a 2×2 lattice has only four sites, so its batches never
reach the break-even and it falls back to rotating term by term.

In [ ]:
def build_qpe_circuit(context, size):
    """Return the full standard-QPE circuit for one lattice size."""
    lattice_operator = build_lattice_operator(size)
    _, _, size_trotter_budget, size_base_time = evolution_schedule(size)

    unitary_builder = AlgorithmRef(
        "hamiltonian_unitary_builder",
        "hubbard_plaquette",
        order=TROTTER_ORDER,
        time=size_base_time,
        t=HOPPING_T,
        u=U_OVER_T * HOPPING_T,
        # Bit k evolves for base_time * 2^k rather than repeating the block 2^k times.
        power_strategy="rescale",
        target_accuracy=size_trotter_budget,
    )
    circuit_builder = create(
        "qpe_circuit_builder",
        "qdk_standard",
        num_bits=QPE_PRECISION_BITS,
        unitary_builder=unitary_builder,
        controlled_circuit_mapper=AlgorithmRef("controlled_circuit_mapper", "hubbard_plaquette"),
    )
    circuit_builder.settings().set("phase_state", "sine")

    with use_qsharp_context(context):
        state_prep = identity_state_prep(num_qubits=lattice_operator.num_qubits)
        return circuit_builder.run(state_prep, lattice_operator)[0]


context = create_qsharp_context()
circuit = build_qpe_circuit(context, EXAMPLE_SIZE)
print(f"built the {EXAMPLE_SIZE}x{EXAMPLE_SIZE} phase-estimation circuit")

## Counting the logical resources

`logical_counts` traces the emitted Q# program and returns the gate counts without
simulating it, so the cost of this step is independent of the number of qubits.

`ccz_count` and `ccix_count` are the two Toffoli-class gates; we report their sum as
`toffolis`. `t_gates` counts only the T gates already present in the traced circuit and
**excludes** the T cost of synthesizing the arbitrary-angle rotations — that cost is
accounted separately in the physical estimate below, which is why `rotations` is reported
as its own column.

In [ ]:
def sample_logical_resources(context, size):
    """Return one row of logical resources for one lattice size."""
    num_sites = size * size
    lattice_operator = build_lattice_operator(size)
    size_energy, size_qpe_budget, size_trotter_budget, size_base_time = evolution_schedule(size)

    built = build_qpe_circuit(context, size)
    qsharp_factory = built._qsharp_factory
    if qsharp_factory is None:
        raise RuntimeError("The QPE circuit does not have Q# factory data.")
    qsharp_context = getattr(qsharp_factory.program, "_qdk_context", qsharp)
    counts = dict(
        qsharp_context.logical_counts(
            qsharp_factory.program,
            *qsharp_factory.parameter.values(),
        )
    )

    ccz_count = int(counts.get("cczCount", 0))
    ccix_count = int(counts.get("ccixCount", 0))
    return {
        "L": size,
        "sites": num_sites,
        "system_qubits": lattice_operator.num_qubits,
        "electrons": round(FILLING * num_sites),
        "target_precision": size_energy,
        "qpe_budget": size_qpe_budget,
        "trotter_budget": size_trotter_budget,
        "qpe_bits": QPE_PRECISION_BITS,
        "num_unitary_queries": 2**QPE_PRECISION_BITS - 1,
        "base_time": size_base_time,
        "t_max": size_base_time * 2**QPE_PRECISION_BITS,
        "power_strategy": "rescale",
        "qpe_error_model": "sine-window-1sigma",
        "qpe_type": "standard-full-circuit",
        "logical_qubits": int(counts["numQubits"]),
        "rotations": int(counts.get("rotationCount", 0)),
        "rotation_depth": int(counts.get("rotationDepth", 0)),
        "t_gates": int(counts.get("tCount", 0)),
        "ccz_count": ccz_count,
        "ccix_count": ccix_count,
        "toffolis": ccz_count + ccix_count,
        "measurements": int(counts.get("measurementCount", 0)),
    }


example_row = sample_logical_resources(context, EXAMPLE_SIZE)
for key in ("logical_qubits", "rotations", "rotation_depth", "t_gates", "toffolis", "measurements"):
    print(f"{key:>16}: {example_row[key]:,}")

## The logical sweep

Sweeping the lattice size shows how each resource scales. The side length must be even for
the plaquette tilings to close on a periodic lattice.

Note that `rotations` *falls* as the lattice grows. That is not a mistake: the error budget
is per site, so a larger lattice is allowed a proportionally larger absolute error, which
lets the Trotter step count per query drop faster than the per-step rotation count rises.

In [ ]:
#: Even side lengths to sweep. Every tower on a 2x2 lattice is below the Hamming-weight
#: break-even, so L=2 is the one row that takes the term-by-term fallback.
LATTICE_SIZES = (2, 4, 6, 8, 10, 20)

frame = pd.DataFrame([sample_logical_resources(context, size) for size in LATTICE_SIZES])
frame[
    [
        "L",
        "sites",
        "system_qubits",
        "logical_qubits",
        "rotations",
        "rotation_depth",
        "t_gates",
        "toffolis",
        "measurements",
    ]
]

`frame` holds the full table, including the error budgets and evolution times. Call
`frame.to_csv("hubbard_logical_resources.csv", index=False)` to save it if you want to diff
two runs against each other.

## Physical resource estimation

The logical counts above say nothing about wall-clock time or physical qubit count. To get
those we map the circuit onto a concrete fault-tolerant architecture: a Majorana qubit with
a $10^{-5}$ physical error rate, error-corrected with a three-auxiliary code, running
lattice surgery, and fed by round-based magic-state factories.

Two knobs are swept:

- **`num_ts_per_rotation`** — how many T gates each arbitrary-angle rotation is synthesized
  into. More T gates per rotation means a more accurate rotation but a larger circuit.
- **`slow_down_factor`** — how far lattice surgery is slowed relative to its fastest
  schedule, trading runtime for fewer magic-state factories and therefore fewer qubits.

`num_ts_per_rotation` deserves particular attention for this circuit. The Hamming-weight
layers apply one synthesized $R_z$ per place value, which puts a large share of the total
cost into the `rotations` column rather than into `t_gates` — so the rotation-synthesis
precision, not the Toffoli count, is what dominates the physical estimate here. Sweeping it
rather than fixing it is what makes the frontier below meaningful.

In [ ]:
from qdk.qre import LatticeSurgery, PSSPC, estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

#: Majorana qubit at a 1e-5 physical error rate.
ARCHITECTURE = Majorana(error_rate=1e-5)

#: Total failure probability allowed for the whole algorithm.
MAX_ERROR = 0.01

application = circuit.get_qre_application()

trace_query = (
    application.q()
    * PSSPC.q(num_ts_per_rotation=list(range(20, 45, 2)))
    * LatticeSurgery.q(slow_down_factor=[1.0 * j for j in range(1, 20)])
)
isa_query = ThreeAux.q() * RoundBasedFactory.q(code_query=ThreeAux.q())

estimates = estimate(
    application,
    ARCHITECTURE,
    isa_query,
    trace_query,
    max_error=MAX_ERROR,
    name=f"Hubbard {EXAMPLE_SIZE}x{EXAMPLE_SIZE}",
)
estimates.add_qubit_partition_column()
estimates.add_factory_summary_column()
estimates.as_frame()

## The runtime / qubit-count frontier

Each point is one feasible configuration. The frontier traces the tradeoff: spending more
physical qubits on magic-state factories shortens the runtime, until the factories stop
being the bottleneck and the curve flattens.

In [ ]:
figure = plot_estimates([estimates], runtime_unit="months", figsize=(15, 7))